# CS448 - Lab 2: Filter Design and Usage

For this lab you will learn how to design some simple filters and how to apply them to solve some common audio problems. Python’s `scipy.signal` package has an extensive set of commands to help you design filters (`firwin`, `firwin2`, `butter`, `cheby1`, `cheby2`,  `ellip`, …), so there is no shortage of options.

In [8]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import scipy.signal as signal

In [9]:
# Make a sound player function that plays array "x" with a sample rate "rate", and labels it with "label"
def sound_player( x, rate=8000, label=''):
    from IPython.display import display, Audio, HTML
    display( HTML( 
    '<style> table, th, td {border: 0px; }</style> <table><tr><td>' + label + 
    '</td><td>' + Audio( x, rate=rate)._repr_html_()[3:] + '</td></tr></table>'
    ))

### Part 0: Warming up with an anti-aliasing filter

You learned in Modue 01 that aliasing is an issue if you use low sample rates to represent a signal that contains high frequency components. As a remedy, you can first apply a low-pass filter to the original signal to kill all the high-frequency components, and then do downsampling. That way, during downsampling, the "ghost" frequencies are not mirrored in the passband because the source of the ghost frequencies are all busted ahead of time. By the way, removing the high-frequency part is okay: the high-frequency part, after downsampling, won't be audible anyway. 

Load ```chirp.wav``` and check out how it sounds and looks in the time-frequency domain. Double-check its sample rate. To see what's going on, you want to lower the sample rate to 11,025 Hz without proper anti-aliasing fitering, by decimating the samples naively. I provided codes for this. Apply a low-pass filter, and then do the same decimation-based downsampling. Play the sounds, observe the spectrograms to see the effect.

You can use ```scipy.signal.firwin``` do design your filter.

Note: If you use a nice resampling routines, something like ```librosa.resample``` they take care of this issue automatically. Wanted to let you know the other people's effort that are not visible to you (yet audible!).

Hint: To apply an FIR filter you can use ```scipy.signal.convolve```.

In [169]:
# load chirp.wav
s, sr=sf.read('chirp.wav', dtype='float32')

# check out the sound and visual. 
# YOUR CODE HERE


# Do a naive downsampling from 44100 Hz to 11025 Hz
target_sr = 11025
factor = sr // target_sr      # 44100 // 11025 = 4 (approx)

# Naive decimation (NO FILTER)
y = s[::factor]


# Check out the sound and visual. 
# YOUR CODE HERE



# Come up with a low-pass filter using firwin and apply it before downsampling
# YOUR CODE HERE

# Check out the sound and visual of the filtered signal. 
# YOUR CODE HERE

# Naive decmiation again, but on the LPF-ed signal  
# YOUR CODE HERE

# Check out the sound and visual of the filtered and downsampled signal. 
# YOUR CODE HERE


## Part 1: Underwater Morse code

You want to review S24-34 in Module 2 to solve this problem. 
```morse.wav``` requires us to extract a Morse code which is buried in environmental noise. Design a filter to bring out the beeps.

Do the following:
- Plot the spectrogram of the given sound and identify the problem
- Figure out the left and right cutoff frequencies
- Design a LP filter by using the *right* cutoff frequency
- A HP filter as well, but this time, using the *left* cutoff frequency
- Verify that they do what the are supposed to do
- Combine the two filters to turn them into a band-pass filter that isolates the Morse beeps. 
- Note: Use ```numpy.sinc``` function and manipulate it by introducing the normalized cutoff frequency ```w```, so you get the sinc function you want. 
- Note: Don't use ```scipy.signal```'s filters. I want you to get to the bottom of this. 
- Note: Don't forget to check out the frequency response of your filters using ```scipy.signal.freqz```. Take care of the ripples using windowing. 

I bugged you guys enough. Now, let's cheat and design an IIR Butterworth filter using ```scipy.signal.butter```. Compare its frequency response to your hand-made one. 

Always check out the filtered signals by listening to them and visualizing them via STFT. 

Hint: To apply an IIR filter you can use ```scipy.signal.lfilter```.

History: Back in the day, when ML classifiers were not good enough, signals had to be *preprocessed* a lot by using things like this, so the noise-sensitlve ML models can understand sound better. Nowadays, it's just done by neural networks altogether, but typically at the cost of additional compute and data curation efforts.  


In [170]:
# Load the sound, play it, and plot its spectrogram
# YOUR CODE HERE

# Define the cutoff frequency and design the LP filter using the sinc function. Check out its shape in the time domain as well as frequency response using freqz.
# YOUR CODE HERE

# Window the sinc function, and see the effect in the STFT domain again. 
# YOUR CODE HERE


# Check out the LPF-ed signal in the audio player and in the frequency domain
# YOUR CODE HERE


# Ditto for HPF
# YOUR CODE HERE

# Now design a band-pass filter by combining the above two filters
# YOUR CODE HERE

# Come up with a Butterworth filter that does the same band-pass filtering, by using signal.butter. Play around with the different order values and see how it affects the frequency response
# YOUR CODE HERE

## Part 3: Say "A"

Let's make the computer talk.  To do so we need two things, an *excitation signal* and a filter.  For the excitation we will use something called the "Rosenberg Pulse". A simple version of it will be the same as a cosine, but it will only be zero between phase $0$ and $3\pi/2$.  It will effectively look like a curvy ramp (or a Shark's fin) from zero to one that takes place during the last quarter of each period. Generate a 0.3 sec long Rosenberg pulse with a frequency of 170Hz, at a sampling rate of 8kHz.  It should sound like a short buzzy tone. Create another one with a frequency of 150 Hz, too. See their waveform representations, too. 

In [171]:
# Make the pulse at 170 Hz and 150 Hz; check out their sound and waveform plots
# YOUR CODE HERE


Now that you have two glottal pulses as the excitation signal, let's apply some filters to generate two vowels, "eh" and "ee".  In the code snippet below I have two tables that hold the center frequency and amplitude of the five resonances that make each vowel.  To model these resonances, design a bandpass filter with a bandwidth of 90Hz centered at each of the given frequencies.  You will end up with five filters for each vowel.  Then take the pulse sound from above and convolve it with the five filters of each vowel (in parallel, not in series) and add the outputs together and scale them with their corresponding amplitudes in the given tables.  Play the results and verify that they sound like a robotic "a" and an "u".

To design the filters, use the ```scipy.signal.firwin``` function with filters of length 61.

In [ ]:
# Formant table for "eh"
Feh = [
    [530, 1840, 2480, 2650, 2900],  # Formant frequencies (Hz)
    [1,    .7,   .6,   .5,   .3],   # Amplitudes
]

# Formant table for "u"
Fee = [
    [270, 2290, 3010, 3300, 3600],  # Formant frequencies (Hz)
    [1,    .7,   .6,   .5,   .3],   # Amplitudes
]

# Apply a sequence of filters to the pulse train and add the results
def vocalize( s, F, sr=8000):
    # s: input sound (the pulse)
    # F: formant table
    
    # YOUR CODE HERE

    
    

# Make the two utterances using the vocalize function and the pulse trains you created in the previous block
# YOUR CODE HERE


# And play them
# YOUR CODE HERE

_IncompleteInputError: incomplete input (3559909154.py, line 27)

Now let's try to compose a word by combining the vowels.  Concatenate the two sounds above and play the result to form the word "A". It will not sound great because the transition from one to the other is too abrupt. To make this a little better, we will need to crossfade the two sounds (just as you have already done for lab 0, part 3).  To do so, use a 200 ms linear crossfade between the two sounds. Play the result and verify that it sounds better. If it sounds more like an "A" who knows if you could get an "A"?

In [173]:
# Concatenate and play the two utterances
# YOUR CODE HERE

# Crossfade them instead
# YOUR CODE HERE

Now take a minute to respect people who source-filter speech synthesis, who had to do this bit-by-bit for entire sentences! :) 